<img src = "../assets/bbai-banner.png" align="center" height = "25%">

# Remaining Useful Life Data Exploration Notebook #

In [ ]:
import pandas as pd

In [ ]:
# Specify the file path
filename = "FD001"
filepath = '../data/training/train_'+filename+'.txt'

# Set up Additional Parameters to be used in data exploration
# Maximum Remaining Useful Life value should be chosen for "no wear" on component.  This should also be chosen to leave enough time to order new component
max_RUL = 130

# Identify the Column Names for the data
column_names = ["id","cycle","setting1","setting2", "setting3",
        "s1","s2","s3","s4","s5","s6","s7","s8","s9","s10","s11","s12","s13","s14","s15","s16","s17","s18","s19","s20","s21",]



In [ ]:
# Load the file into Pandas dataframe
data = pd.read_csv(filepath, sep=" ", header=None)
data.drop(data.columns[[26, 27]], axis=1, inplace=True)
data.columns = column_names

# Show me the data
data.head()

In [ ]:
print("Data Shape: ", data.shape)
print("\tRows : ", data.shape[0])
print("\tColumns : ", data.shape[1])  

### Now that we have the data loaded, there are a few things that we need to do with it.
- Make sure the data is sorted by ID and Cycle
- Determine the maximum Cycle for each ID.  
- In training data, the RUL is $ Max Cycle - Current Cycle $, so we will set that as a column
- We will limit the RUL to the maximum RUL specified above
- We will set **R_early** column to the RUL in training data

In [ ]:
data = data.sort_values(["id", "cycle"])
data.head()

Now the author claims to Preprocess the data for piece-wise RUL.  

What is happening is that the datasets contain information for ID until failure with each cycle being a measurement.  The maximum cycle is the end of life of that part.

The following code adds the **maximum columns** and the **RUL column** as *transient columns* and adds the **R_early column**.

In [ ]:
rul = pd.DataFrame(data.groupby("id")["cycle"].max()).reset_index()
rul.columns = ["id", "max"]
rul.head()

data = data.merge(rul, on=["id"], how="left")
data.head()
data.tail()


In [ ]:
data["RUL"] = data["max"] - data["cycle"]
data.drop("max", axis=1, inplace=True)
data.head()


In [ ]:
data.tail()

In [ ]:
data["R_early"]=data["RUL"].apply(lambda x: max_RUL if x>= max_RUL else x)
data.drop("RUL", axis=1, inplace=True)
data.head()


In [ ]:
data.tail()